# IronyLab в Google Colab
Учебная, объяснимая оценка **иронического потенциала**, а не доказательство намерения автора. Выполняйте ячейки сверху вниз (`Shift+Enter`).

In [ ]:
import re, json, csv, io
from dataclasses import dataclass, asdict

PHRASES = {
    'ну да': 14, 'ну конечно': 20, 'да конечно': 18, 'ага конечно': 22,
    'как же': 18, 'вот спасибо': 16, 'просто прекрасно': 17,
    'просто замечательно': 17, 'гениально': 9, 'блестяще': 9,
    'yeah right': 22, 'as if': 20, 'thanks a lot': 15, 'just great': 17,
}
POSITIVE = set('прекрасно прекрасный прекрасная прекрасное прекрасные замечательно замечательный замечательная отлично отличный великолепно гениально блестяще люблю обожаю спасибо счастье great wonderful excellent brilliant love perfect thanks'.split())
NEGATIVE = set('опять сломался сломалась сломано ошибка провал ужас ненавижу дождь ливень опоздал опоздала проблема плохо again broken error failure awful hate late problem bad'.split())

def analyze(text):
    text = ' '.join(str(text).split()); low = text.lower()
    words = set(re.findall(r'[a-zа-яё]+', low, flags=re.I))
    signals = []
    for phrase, weight in PHRASES.items():
        if phrase in low: signals.append({'name':'лексическая формула','evidence':phrase,'weight':weight})
    pos, neg = sorted(words & POSITIVE), sorted(words & NEGATIVE)
    if pos and neg: signals.append({'name':'оценочный контраст','evidence':f'{pos[0]} ↔ {neg[0]}','weight':28})
    if re.search(r'[!?]{2,}', text): signals.append({'name':'экспрессивная пунктуация','evidence':'!!/??','weight':7})
    if re.search(r'🙃|😏|😉|🤡|👏', text): signals.append({'name':'эмодзи','evidence':'эмодзи','weight':13})
    bonus = max(0, len({x['name'] for x in signals}) - 1) * 3
    score = min(100, sum(x['weight'] for x in signals) + bonus)
    level = 'низкий' if score < 20 else 'умеренный' if score < 45 else 'повышенный' if score < 70 else 'высокий'
    return {'text':text, 'score':score, 'level':level, 'signals':signals}

print('IronyLab готов')

## 1. Проверяем отдельные примеры

In [ ]:
examples = [
    'Заседание начинается в десять часов.',
    'Прекрасно, опять сломался принтер!',
    'Ну да, конечно, просто замечательно!!! 🙃',
]
for example in examples:
    result = analyze(example)
    print(f"{result['score']:>3}/100  {result['level']:<10} {example}")
    print('   ', result['signals'])

## 2. Собственный эксперимент
Измените фразы. Затем попробуйте добавить новый маркер в `PHRASES` и повторно выполнить ячейки.

In [ ]:
my_text = 'Вот спасибо, поезд снова опоздал!'
analyze(my_text)

## 3. Загружаем TXT или CSV
Для CSV нужен столбец `text`.

In [ ]:
from google.colab import files
uploaded = files.upload()
name, content = next(iter(uploaded.items()))
if name.lower().endswith('.csv'):
    rows = list(csv.DictReader(io.StringIO(content.decode('utf-8-sig'))))
    if not rows or 'text' not in rows[0]: raise ValueError('Нужен столбец text')
    results = [dict(row, **{k:v for k,v in analyze(row['text']).items() if k in ('score','level')}) for row in rows]
else:
    results = [analyze(content.decode('utf-8'))]
results[:5]

## 4. Скачиваем результат

In [ ]:
if name.lower().endswith('.csv'):
    out_name = 'ironylab_results.csv'
    with open(out_name, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys()); writer.writeheader(); writer.writerows(results)
else:
    out_name = 'ironylab_result.json'
    with open(out_name, 'w', encoding='utf-8') as f: json.dump(results[0], f, ensure_ascii=False, indent=2)
files.download(out_name)